# ocr enabled pdf analyser

In [ ]:
import os
import base64
import fitz 
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
import io
from PIL import Image

In [ ]:
vllm = ChatOllama(model="qwen2.5vl:3b", temperature=0)

def pdf_to_text(pdf_path):
    pdf = fitz.open(pdf_path)
    base64_images = []
    for p_num in range(len(pdf)):
        page = pdf.load_page(p_num)
        zoom = 1.0
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat)

        img_bytes = pix.tobytes("png")
        b64_encoded = base64.b64encode(img_bytes).decode('utf-8')
        base64_images.append(b64_encoded)

    print(f"converted {len(base64_images)} pages to images")
    return base64_images

pdf_to_text("table.pdf")

In [ ]:
from langchain_core.messages import HumanMessage    

def txt_from_img(b64_img):
    full_markdown_text = ""

    print(f"starting ocr extraction for {len(b64_img)} images...")

    for i, b64img in enumerate(b64_img):
        print(f"reading page{i + 1}...")

        message = HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": """You are an expert OCR system. Extract all text, data, and tables from this document image. 
                    Format your output as clean Markdown. Preserve all structure, bullet points, and table formatting. 
                    Do NOT add any conversational filler (like 'Here is the text:'), just output the extracted content."""
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{b64img}"}
                }
            ]
        )

        ans = vllm.invoke([message])

        full_markdown_text += f"\n\n--- Page {i + 1} ---\n\n"
        full_markdown_text += ans.content

    print("OCR extraction complete.")
    return full_markdown_text




In [ ]:
images = pdf_to_text("table.pdf")

extracted = txt_from_img(images)

with open("extracted_text.md", "w", encoding="utf-8") as f:
    f.write(extracted)

print("Extracted text saved to extracted_text.md")

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding


Settings.llm = Ollama(model="qwen3:4b", request_timeout=120.0,
    additional_kwargs={"num_ctx": 4096})
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

documents = SimpleDirectoryReader(input_files=["extracted_text.md"]).load_data()

parser = MarkdownNodeParser()
nodes = parser.get_nodes_from_documents(documents)
print(f"parsed document into {len(nodes)} structural nodes.")

print("Embedding data ...")
index = VectorStoreIndex(nodes)

query_engine = index.as_query_engine()
print("Ready to chat with your document!")

In [ ]:
question = "What are the main topics or data points mentioned in the document?"

response = query_engine.query(question)

print(f"**Question:** {question}\n")
print(f"**Answer:**\n{response.response}")